<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_08-1_Minimal_Agent_Harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 08-1 | Minimal Agent Harness 구현

**학습 목표**: Model의 행동 선택, Tool 실행, State·Memory 갱신, Orchestration, 승인·예산 제한, 로그의 책임을 구분하고 직접 구현합니다.

- 실습 시나리오: 학생이 **지능시스템 강의 정책**에 관해 질문하면 모의 검색 도구로 답합니다.
- 실제 LLM이나 외부 API는 사용하지 않습니다. `policy()` 함수가 모델의 다음 행동 결정을 **규칙으로 흉내**냅니다.
- **Google Colab에서 설치 없이 실행**할 수 있습니다. 위에서 아래로 셀을 순서대로 실행하세요.

예제 강의 정책은 실제 학교 규정을 조회한 값이 아닌 **모의 데이터**입니다.

## 1. 핵심 구조

`사용자 질문 → policy(Model 역할) → Harness(검증·승인·예산 검사) → Tool 실행 → 결과(Observation) → State·Memory 갱신 → 다음 행동 또는 종료`

**누가 무슨 일을 하는가?**

| 구성요소 | 하는 일 |
|---|---|
| Model 역할(`policy`) | 어떤 도구를 사용할지, 언제 답을 완성할지 선택 |
| Tool | 강의 정책 문서에서 내용을 검색 |
| State | 이번 실행에서 관찰한 결과와 현재 진행 상황 기록 |
| Memory | 실행 중 확인한 검색 결과 기록(이번 예제에서는 임시 메모리) |
| Harness / Orchestration | 검사, 승인, 예산 제한, Tool 호출, 로그, 반복과 종료를 관리 |

이 실습의 Memory는 **장기 기억이 아니라 실행 중 메모리**입니다. 장기 저장과 Checkpoint는 다른 실습에서 다룹니다.

## 2. 예제 데이터와 검색 Tool

In [1]:
# 외부 웹사이트에 접속하지 않고 연습하기 위해 만든 작은 모의 문서입니다.
# 실제 대학 규정을 의미하지 않습니다.
POLICY_DOCUMENTS = [
    {"id": "doc-01", "title": "평가 안내", "text": "지능시스템 모의 수업의 과제 반영 비율은 20%입니다."},
    {"id": "doc-02", "title": "출석 안내", "text": "지능시스템 모의 수업의 출석 반영 비율은 20%입니다."},
    {"id": "doc-03", "title": "AI 활용 안내", "text": "모의 과제에서는 AI를 활용한 경우 활용 사실과 출처를 표시합니다."},
]

# Tool을 실행하는 함수입니다. 입력으로 검색어(query)를 받습니다.
def course_search_policy(query: str) -> dict:
    """문서 제목 또는 본문에 검색어가 들어 있는 항목을 찾습니다."""
    # 검색어를 검사합니다. 빈 문자열은 올바른 검색이 아닙니다.
    if not isinstance(query, str) or not query.strip():
        return {"error_code": "INVALID_ARGUMENT", "message": "검색어를 입력하세요."}

    # 초보자도 이해하기 쉬운 단순 부분 문자열 검색입니다.
    # 실제 검색 서비스는 형태소 분석, 임베딩 등을 활용할 수 있습니다.
    matches = []
    for document in POLICY_DOCUMENTS:
        if query.strip() in document["title"] or query.strip() in document["text"]:
            matches.append(document)

    # 다음 행동을 결정하기 쉬운 구조로 결과를 반환합니다.
    return {"query": query, "count": len(matches), "results": matches}

print(course_search_policy("과제"))

{'query': '과제', 'count': 2, 'results': [{'id': 'doc-01', 'title': '평가 안내', 'text': '지능시스템 모의 수업의 과제 반영 비율은 20%입니다.'}, {'id': 'doc-03', 'title': 'AI 활용 안내', 'text': '모의 과제에서는 AI를 활용한 경우 활용 사실과 출처를 표시합니다.'}]}


## 3. Tool 목록과 권한 정보

In [2]:
# Harness가 사용하는 Tool Registry(도구 목록)입니다.
# 각 도구의 실행 함수와 권한 정보를 함께 저장합니다.
# requires_approval=True라면 사람이 허락해야 실행됩니다.

def kb_update(note: str) -> dict:
    """실습용 수정 Tool입니다. 실제 파일이나 DB를 변경하지 않습니다."""
    return {"status": "updated_demo_only", "note": note}

TOOLS = {
    "course_search_policy": {
        "fn": course_search_policy,
        "read_only": True,
        "requires_approval": False,
    },
    "kb_update": {
        "fn": kb_update,
        "read_only": False,
        "requires_approval": True,
    },
}

print("등록된 Tool:", list(TOOLS))

등록된 Tool: ['course_search_policy', 'kb_update']


## 4. State를 보조하는 Memory와 Budget

In [3]:
class AgentMemory:
    """이번 실행에서 Tool 결과를 저장하는 아주 단순한 임시 메모리입니다."""

    def __init__(self):
        self.items = []  # 검색 결과를 저장할 빈 리스트

    def observe(self, tool_name: str, result: dict):
        # Tool 이름과 결과를 하나의 기록으로 묶습니다.
        self.items.append({"tool": tool_name, "result": result})


class Budget:
    """실행 횟수·Tool 호출 횟수에 제한을 두는 클래스입니다."""

    def __init__(self, max_steps=5, max_tool_calls=3):
        self.max_steps = max_steps
        self.max_tool_calls = max_tool_calls
        self.steps = 0
        self.tool_calls = 0

    def can_take_step(self):
        # 이미 정해진 횟수만큼 행동했다면 더 실행하지 않습니다.
        return self.steps < self.max_steps

    def can_call_tool(self):
        # 도구 호출 예산은 실행하기 전에 확인해야 합니다.
        return self.tool_calls < self.max_tool_calls

    def record_step(self):
        self.steps += 1

    def record_tool_call(self):
        self.tool_calls += 1


memory_example = AgentMemory()
budget_example = Budget(max_steps=4, max_tool_calls=2)
print("초기 메모리:", memory_example.items)
print("최대 행동 횟수:", budget_example.max_steps)

초기 메모리: []
최대 행동 횟수: 4


## 5. Model 역할을 흉내 내는 `policy(state)`

규칙은 간단합니다.

1. 검색 결과가 없으면 검색 Tool을 선택합니다.
2. 검색 결과가 있으면 그 내용을 정리해 최종 답변을 만듭니다.
3. 검색 결과가 0개이면 근거가 없다고 답변합니다.

**중요:** 실제 LLM이 판단하는 것이 아닙니다. 검색어를 고르는 규칙은 실습의 편의를 위한 것입니다.

In [4]:
def policy(state: dict) -> dict:
    """현재 State를 읽고 다음 행동(action)을 정합니다."""
    # 아직 Tool 결과를 받은 적이 없으면 검색부터 수행합니다.
    if not state["observations"]:
        question = state["goal"]
        # 사용자의 질문에서 간단한 검색어를 고릅니다.
        if "과제" in question:
            query = "과제"
        elif "출석" in question:
            query = "출석"
        elif "AI" in question:
            query = "AI"
        else:
            query = question  # 모르는 질문은 문장 전체로 검색합니다.
        return {"tool": "course_search_policy", "args": {"query": query}}

    # 가장 최근 검색 결과를 읽고 답을 완성합니다.
    last_result = state["observations"][-1]
    if "error_code" in last_result:
        answer = f"검색 중 오류: {last_result['message']}"
    elif last_result["count"] == 0:
        answer = "모의 정책 문서에서 관련 근거를 찾지 못했습니다."
    else:
        # 검색된 문서의 본문만 이어 붙입니다.
        answer = " ".join(doc["text"] for doc in last_result["results"])

    return {"tool": "finish", "answer": answer}

print(policy({"goal": "과제 비율이 궁금해요", "observations": []}))

{'tool': 'course_search_policy', 'args': {'query': '과제'}}


## 6. 핵심: Agent Harness 실행 루프

In [5]:
def run_agent(goal, policy_fn, tools, memory, budget, approve):
    """정책 선택 → 검사 → 실행 → 관찰 → 반복을 수행하는 간단한 Harness입니다."""
    # State는 '이번 실행'에만 필요한 정보를 담습니다.
    state = {"goal": goal, "observations": []}
    log = []  # 나중에 문제가 생겼을 때 어느 단계에서 일어났는지 확인합니다.

    while True:
        # [1] 무한 반복 방지: 행동 횟수 한도를 먼저 확인합니다.
        if not budget.can_take_step():
            log.append({"event": "halt", "reason": "max_steps"})
            break

        # [2] Model 역할: 다음 행동을 고릅니다.
        action = policy_fn(state)
        budget.record_step()

        # [3] 종료 행동이면 결과를 기록하고 반복문을 끝냅니다.
        if action["tool"] == "finish":
            log.append({"event": "finish", "answer": action["answer"]})
            break

        tool_name = action["tool"]
        tool = tools.get(tool_name)

        # [4] 등록되지 않은 Tool은 실행하지 않습니다.
        if tool is None:
            log.append({"event": "error", "reason": "invalid_tool", "tool": tool_name})
            break

        # [5] 수정·삭제와 같이 승인이 필요한 Tool은 먼저 승인을 받습니다.
        if tool["requires_approval"] and not approve(action):
            log.append({"event": "blocked", "reason": "human_approval_required", "tool": tool_name})
            break  # 실습에서는 승인이 거부되면 안전하게 종료합니다.

        # [6] Tool을 실제로 실행하기 '전'에 호출 예산을 확인합니다.
        if not budget.can_call_tool():
            log.append({"event": "halt", "reason": "max_tool_calls"})
            break

        # [7] Tool 호출: 쉬운 예제를 위해 인자를 그대로 함수에 전달합니다.
        try:
            result = tool["fn"](**action.get("args", {}))
        except (TypeError, ValueError) as error:
            # 잘못된 인자를 넣어도 Notebook 전체가 중단되지 않게 처리합니다.
            log.append({"event": "error", "reason": "invalid_argument", "message": str(error)})
            break
        except Exception as error:
            # 실제 서비스에서는 내부 오류 정보를 사용자에게 그대로 노출하지 않습니다.
            log.append({"event": "error", "reason": "tool_execution_failed", "type": type(error).__name__})
            break

        budget.record_tool_call()

        # [8] 검색 결과(Observation)를 메모리와 상태에 반영합니다.
        memory.observe(tool_name, result)
        state["observations"].append(result)
        log.append({"event": "tool", "tool": tool_name, "result": result})
        # 다음 반복에서 policy_fn은 새 State를 보고 다음 행동을 고릅니다.

    # 실습 결과를 한 번에 살펴보기 위해 여러 값을 반환합니다.
    return {"state": state, "log": log, "memory": memory.items,
            "steps": budget.steps, "tool_calls": budget.tool_calls}


def deny_approval(action):
    """사람이 승인을 거부했다고 가정하는 테스트 함수입니다."""
    return False


def allow_approval(action):
    """사람이 승인했다고 가정합니다. 실제 서비스에서는 별도 확인 절차가 필요합니다."""
    return True

## 7. 정상 시나리오: 정책 검색 후 답변

In [6]:
# 새로운 질문마다 새 Memory와 새 Budget을 만듭니다.
normal_run = run_agent(
    goal="지능시스템 과제 반영 비율은 얼마인가요?",
    policy_fn=policy,
    tools=TOOLS,
    memory=AgentMemory(),
    budget=Budget(max_steps=5, max_tool_calls=3),
    approve=deny_approval,
)

# 핵심 결과만 출력해 확인합니다.
for entry in normal_run["log"]:
    print("이벤트:", entry["event"])
    if entry["event"] == "finish":
        print("최종 답변:", entry["answer"])
print("Tool 호출 횟수:", normal_run["tool_calls"])
print("Memory 항목 수:", len(normal_run["memory"]))

이벤트: tool
이벤트: finish
최종 답변: 지능시스템 모의 수업의 과제 반영 비율은 20%입니다. 모의 과제에서는 AI를 활용한 경우 활용 사실과 출처를 표시합니다.
Tool 호출 횟수: 1
Memory 항목 수: 1


## 8. 실패 시나리오 ①: 승인되지 않은 수정 차단

In [7]:
# 이 policy는 처음부터 수정 Tool을 실행해 달라고 요청합니다.
# Model이 이런 결정을 하더라도 Harness가 허용 여부를 확인해야 합니다.
def request_update(state):
    return {"tool": "kb_update", "args": {"note": "정책을 수정해 주세요"}}

approval_run = run_agent(
    goal="강의 정책 수정",
    policy_fn=request_update,
    tools=TOOLS,
    memory=AgentMemory(),
    budget=Budget(max_steps=3, max_tool_calls=2),
    approve=deny_approval,
)
print(approval_run["log"])
print("실제 Tool 호출 횟수:", approval_run["tool_calls"])
# 기대 결과: blocked, Tool 호출 횟수 0

[{'event': 'blocked', 'reason': 'human_approval_required', 'tool': 'kb_update'}]
실제 Tool 호출 횟수: 0


## 9. 실패 시나리오 ②: 존재하지 않는 Tool 차단

In [8]:
def request_unknown_tool(state):
    return {"tool": "delete_everything", "args": {}}

invalid_run = run_agent(
    goal="등록되지 않은 도구 실행",
    policy_fn=request_unknown_tool,
    tools=TOOLS,
    memory=AgentMemory(),
    budget=Budget(),
    approve=deny_approval,
)
print(invalid_run["log"])
# 기대 결과: invalid_tool

[{'event': 'error', 'reason': 'invalid_tool', 'tool': 'delete_everything'}]


## 10. 실패 시나리오 ③: Budget 초과 시 중단

In [9]:
# 항상 검색만 시도하는 잘못된 정책입니다.
# 종료를 선택하지 않으므로 Harness가 호출 횟수를 제한해야 합니다.
def repeat_search(state):
    return {"tool": "course_search_policy", "args": {"query": "과제"}}

budget_run = run_agent(
    goal="검색 반복",
    policy_fn=repeat_search,
    tools=TOOLS,
    memory=AgentMemory(),
    budget=Budget(max_steps=10, max_tool_calls=2),
    approve=deny_approval,
)
print("이벤트 순서:", [entry["event"] for entry in budget_run["log"]])
print("마지막 로그:", budget_run["log"][-1])
print("실제 호출 횟수:", budget_run["tool_calls"])
# 기대 결과: 검색 2회만 실행하고 max_tool_calls로 중단

이벤트 순서: ['tool', 'tool', 'halt']
마지막 로그: {'event': 'halt', 'reason': 'max_tool_calls'}
실제 호출 횟수: 2


## 11. 간단한 평가(Eval): 시나리오별 성공 여부 확인

In [10]:
# 정답뿐 아니라 '실행이 안전하게 제어되었는지'도 확인해야 합니다.
# 각 결과에서 마지막 event와 이유를 확인합니다.
checks = {
    "정상 검색": any(item["event"] == "finish" for item in normal_run["log"]),
    "승인 없이 수정 차단": approval_run["log"][-1]["event"] == "blocked" and approval_run["tool_calls"] == 0,
    "알 수 없는 Tool 차단": invalid_run["log"][-1].get("reason") == "invalid_tool",
    "예산 초과 차단": budget_run["log"][-1].get("reason") == "max_tool_calls" and budget_run["tool_calls"] == 2,
}

for test_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {test_name}")

# 테스트 실패가 있으면 원인을 바로 확인하도록 합니다.
assert all(checks.values()), "일부 테스트가 실패했습니다. 이전 셀부터 다시 확인하세요."
print("모든 기본 테스트를 통과했습니다!")

PASS | 정상 검색
PASS | 승인 없이 수정 차단
PASS | 알 수 없는 Tool 차단
PASS | 예산 초과 차단
모든 기본 테스트를 통과했습니다!


### 정리

- **Model 역할**: 다음 행동 선택
- **Tool**: 실제 작업 수행
- **State·Memory**: 실행 결과 보관
- **Harness**: 검증·승인·예산·반복·종료·로그 관리

> 참고: 이 예제는 원리를 보여주기 위한 교육용입니다. 실제 운영에서는 Tool 입력 스키마 검증, 별도 인증·권한 검사, 영속적 로그, 타임아웃, 재시도·멱등성, 체크포인트 등이 더 필요합니다.